# ATLAS: Attention U-Net for Brain Tumor Segmentation - BraTS 2020
### DL Sem 7 - Colab T4 Training Notebook

**Run on Colab:** Runtime -> Change runtime type -> T4 GPU -> Run All (Ctrl+F9)  
Trains 50 epochs in ~2.5 hrs | Inference + plots in same notebook


### 0. Setup & Check GPU

In [ ]:
!pip install -q albumentations segmentation-models-pytorch nibabel tqdm torchinfo 2>&1 | tail -n 1
import torch, random, numpy as np
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
else:
    print("WARNING: No GPU - switch to T4 in Runtime settings")

# seed for reproducibility
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


### 1. Download BraTS 2020
Uses Kaggle API if `kaggle.json` uploaded, else falls back to sliced .npy (already in repo for demo).

In [ ]:
import os, glob
# Option A: with kaggle.json (uncomment if you have it)
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d awsaf49/brats20-dataset-training-validation -p data --unzip -q

# Option B: use helper script (creates dummy slices if Kaggle not configured - for testing pipeline)
!python data/download.py

print(os.listdir("data"))
print("train:", len(glob.glob("data/brats20/train/images/*.npy")), "val:", len(glob.glob("data/brats20/val/images/*.npy")))

### 2. Dataset - Visualize a sample

In [ ]:
import sys; sys.path.append("src")
from dataset import BraTSDataset, get_transforms
import matplotlib.pyplot as plt
import numpy as np

ds = BraTSDataset("data/brats20", split="train", transform=get_transforms(True), img_size=128)
print(f"Dataset size: {len(ds)}")
img, mask = ds[10]
print(f"img: {img.shape} mask: {mask.shape} | img min/max {img.min():.2f}/{img.max():.2f}")

fig, ax = plt.subplots(1,5, figsize=(14,3))
titles=["FLAIR","T1","T1ce","T2","Mask"]
for i in range(4):
    ax[i].imshow(img[i], cmap="gray"); ax[i].set_title(titles[i]); ax[i].axis("off")
ax[4].imshow(mask[0], cmap="gray"); ax[4].set_title("GT Mask"); ax[4].axis("off")
plt.tight_layout(); plt.show()

### 3. Model - Attention U-Net
Same as `src/model.py` - Attention Gates + DoubleConv + Dropout

In [ ]:
from model import get_model
from utils import count_params

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_model("attention_unet").to(device)
print(f"Params: {count_params(model):,}")
print(model)

# dummy forward check
x = torch.randn(2,4,128,128).to(device)
with torch.no_grad():
    y = model(x)
print(f"forward: {x.shape} -> {y.shape}")

### 4. Training Loop (with AMP + Early Stopping)
Same logic as `src/train.py` but inline so outputs show in Colab.

In [ ]:
from torch.utils.data import DataLoader
from utils import DiceFocalLoss, dice_coef, iou_score
from tqdm import tqdm
import csv, time

train_ds = BraTSDataset("data/brats20", "train", transform=get_transforms(True), img_size=128)
val_ds = BraTSDataset("data/brats20", "val", transform=get_transforms(False), img_size=128)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

criterion = DiceFocalLoss(dice_weight=0.6)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)
scaler = torch.cuda.amp.GradScaler() if device.type=="cuda" else None

EPOCHS=50
best_dice=0
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results", exist_ok=True)

# csv logger
log_path="results/metrics.csv"
with open(log_path,'w',newline='') as f:
    csv.writer(f).writerow(["epoch","train_loss","val_loss","val_dice","val_iou","lr","time_min"])

for epoch in range(1, EPOCHS+1):
    t0=time.time()
    # ---- train ----
    model.train(); train_loss=0
    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]", leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                preds = model(imgs)
                loss = criterion(preds, masks)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            preds = model(imgs)
            loss = criterion(preds, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        train_loss+=loss.item()
    train_loss/=len(train_loader)
    
    # ---- val ----
    model.eval(); val_loss=val_dice=val_iou=0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            val_loss+=criterion(preds, masks).item()
            val_dice+=dice_coef(preds, masks).item()
            val_iou+=iou_score(preds, masks).item()
    val_loss/=len(val_loader); val_dice/=len(val_loader); val_iou/=len(val_loader)
    scheduler.step(val_dice)
    lr=optimizer.param_groups[0]['lr']
    mins=(time.time()-t0)/60
    print(f"Epoch {epoch:02d}/{EPOCHS} | train {train_loss:.4f} | val {val_loss:.4f} | dice {val_dice:.4f} | iou {val_iou:.4f} | lr {lr:.6f} | {mins:.1f} min")
    with open(log_path,'a',newline='') as f:
        csv.writer(f).writerow([epoch,f"{train_loss:.4f}",f"{val_loss:.4f}",f"{val_dice:.4f}",f"{val_iou:.4f}",f"{lr:.6f}",f"{mins:.2f}"])
    if val_dice>best_dice:
        best_dice=val_dice
        torch.save({"epoch":epoch,"model_state":model.state_dict(),"best_dice":best_dice}, "checkpoints/best_model.pth")
        print(f"  -> saved best {best_dice:.4f}")

print(f"Done. Best dice {best_dice:.4f}")

### 5. Plots - Loss / Dice curves

In [ ]:
import pandas as pd
df=pd.read_csv("results/metrics.csv")
df.head(10)
print(df.tail())
print(f"Best dice row:\n{df.loc[df['val_dice'].idxmax()]}")

fig, ax=plt.subplots(1,2, figsize=(13,4))
ax[0].plot(df['epoch'], df['train_loss'], label='train'); ax[0].plot(df['epoch'], df['val_loss'], label='val')
ax[0].set_title('Loss'); ax[0].set_xlabel('Epoch'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(df['epoch'], df['val_dice'], label='dice'); ax[1].plot(df['epoch'], df['val_iou'], label='iou')
ax[1].set_title('Dice / IoU'); ax[1].set_xlabel('Epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig("results/curves.png", dpi=150); plt.show()

### 6. Visualize Predictions vs Ground Truth

In [ ]:
model.eval()
imgs, masks = next(iter(val_loader))
imgs, masks = imgs.to(device), masks.to(device)
with torch.no_grad():
    preds = model(imgs)
preds_bin = (preds>0.5).float().cpu().numpy()
masks = masks.cpu().numpy()
imgs = imgs.cpu().numpy()

fig, axes=plt.subplots(3,4, figsize=(12,8))
for i in range(4):
    axes[0,i].imshow(imgs[i,0], cmap='gray'); axes[0,i].set_title(f'FLAIR {i+1}'); axes[0,i].axis('off')
    axes[1,i].imshow(masks[i,0], cmap='gray'); axes[1,i].set_title('Ground Truth'); axes[1,i].axis('off')
    axes[2,i].imshow(preds_bin[i,0], cmap='gray'); axes[2,i].set_title('Prediction'); axes[2,i].axis('off')
plt.tight_layout(); plt.savefig("results/sample_predictions.png", dpi=150); plt.show()

# dice for these 4 samples
for i in range(4):
    inter=(preds_bin[i,0]*masks[i,0]).sum()
    dice=2*inter/(preds_bin[i,0].sum()+masks[i,0].sum()+1e-6)
    print(f"sample {i}: dice {dice:.4f}")

### 7. Save for GitHub (optional)
After Colab, download `results/` + `checkpoints/best_model.pth` and push to repo.

In [ ]:
from google.colab import files
# !zip -r results.zip results checkpoints/best_model.pth
# files.download('results.zip')
print("Done - download results/curves.png and sample_predictions.png for PPT")